In [1]:
from mmml.interfaces.dcmInterface import (
    run_kernel_fit_pipeline,
    fit_kernel_from_training_data,
    predict_charges_from_kernel,
    evaluate_and_write_h5,
    optimize_charge_positions,
)
fn = "/home/ericb/mmml_tutorial/cli/charmm_ml_comparison/charmm_ml_comparison.h5"

# Full pipeline (optional ESP-based optimization → kernel fit → CHARMM files + H5)
result = run_kernel_fit_pipeline(
    h5_path=fn,
    out_dir="kernel_out",
    out_h5="kernel_eval.h5",
    out_mdcm="meoh.mdcm",   # optional, default: kernel_out/MEOH.mdcm
    out_kmdcm="meoh.kmdcm", # optional, default: kernel_out/MEOH.kmdcm
    optimize_positions=True,
    residue_name="MEOH",
)

# result["out_mdcm_path"], result["out_kmdcm_path"]

In [2]:
result

{'X_fit': array([[1.42580927, 1.95512867, 1.14246269, 1.14137835, 1.12141787,
         0.96874555, 2.2788343 , 1.91375576, 2.08134849, 2.50790051,
         2.17432538, 2.87785361, 1.86971617, 1.86969088, 1.77121734],
        [1.43353708, 1.79578011, 1.11026864, 1.14177557, 1.1331624 ,
         0.99485199, 2.08006859, 2.18989974, 1.99462491, 2.11227855,
         2.33178034, 2.71056799, 1.90033685, 1.97482099, 1.62061664],
        [1.42995239, 1.98558992, 1.33033987, 0.91496583, 1.11059125,
         0.96847338, 2.18702488, 1.89359714, 2.12259807, 2.45645383,
         2.2655654 , 2.92421854, 2.03910361, 1.89844072, 1.58300602],
        [1.43562704, 1.80140468, 1.12676092, 1.12604825, 1.12137115,
         0.68063166, 2.04365138, 2.04221461, 2.14267621, 2.23676895,
         2.24351165, 2.7019946 , 2.03402736, 1.74760213, 1.75018151],
        [1.42557957, 1.97430516, 1.3190004 , 0.90433027, 1.10733341,
         1.00238629, 2.23906499, 1.95477091, 2.07478998, 2.31408749,
         2.3743259 , 

In [3]:
import ase
from ase import io
import mmml
from mmml.interfaces.pycharmmInterface import import_pycharmm
import os
import sys
from pathlib import Path

from mmml.cli.make.make_res import main_loop
import argparse
import pycharmm

/home/ericb/mmml/mmml/data/charmm/top_all36_cgenff.rtf
/home/ericb/mmml/mmml/data/charmm/par_all36_cgenff.prm
CHARMM_HOME /home/ericb/mmml/setup/charmm
CHARMM_LIB_DIR /home/ericb/mmml/setup/charmm
  
 CHARMM>     BLOCK
 WARNING from DECODI -- Zero length string being converted to 0
 Block structure initialized with   3 blocks.
 All atoms have been assigned to block 1.
 All interaction coefficients have been set to unity.
  Setting number of block exclusions nblock_excldPairs=0
  
  BLOCK>            CALL 1 SELE ALL END
 SELRPN>      0 atoms have been selected out of      0
 The selected atoms have been reassigned to block   1
  
  BLOCK>              COEFF 1 1 1.0
  
  BLOCK>            END
 Matrix of Interaction Coefficients
 
    1.00000
    1.00000   1.00000
    1.00000   1.00000   1.00000
 Matrix of BOND Interaction Coefficients
 
    1.00000
    1.00000   1.00000
    1.00000   1.00000   1.00000
 Matrix of ANGLE Interaction Coefficients
 
    1.00000
    1.00000   1.00000
    1.000

In [4]:


def main():
    args = argparse.Namespace(res="MEOH", skip_energy_show=True)
    print("=== 01: make_res programmatic ===")
    atoms = main_loop(args)
    print(f"Generated {len(atoms)} atoms")
    print("Output: pdb/initial.pdb, psf/initial.psf, xyz/initial.xyz, CHARMM topology files")


output =    main()


from pycharmm.coor import set_positions, get_positions


=== 01: make_res programmatic ===
CHARMM_HOME:  /home/ericb/mmml/setup/charmm
CHARMM_LIB_DIR:  /home/ericb/mmml/setup/charmm
***** Generating residue from residue name (MEOH) *****
***** Generating residue *****

 DRUDES PARTICLES WILL BE GENERATED AUTOMATICALLY FOR ALL ATOMS WITH NON-ZERO ALPHA
 Thole-type dipole screening, Slater-Delta shape {S(u) = 1 - (1+u/2)*exp(-u)}, default radius =  1.300000
***** Generating coordinates *****

          COORDINATE FILE MODULE
         6  EXT
         1         1  MEOH      CB           9999.0000000000     9999.0000000000     9999.0000000000  MEOH      1               0.0000000000
         2         1  MEOH      OG           9999.0000000000     9999.0000000000     9999.0000000000  MEOH      1               0.0000000000
         3         1  MEOH      HG1          9999.0000000000     9999.0000000000     9999.0000000000  MEOH      1               0.0000000000
         4         1  MEOH      HB1          9999.0000000000     9999.0000000000     9999

In [ ]:

dcm_script = """
open unit 11 card read name meoh.mdcm
open unit 12 card read name meoh.kmdcm
open unit 99 write card name dcm.xyz
!DCM IUDCM 11 TSHIFT XYZ 99
DCM KERN 12 IUDCM 11 TSHIFT XYZ 99
"""
pycharmm.lingo.charmm_script(dcm_script)
pycharmm.lingo.charmm_script("ENER")
pycharmm.lingo.charmm_script("STOP")
